# CGRA Solutions — Chapter 3 (Ex 3.1–3.7)
Shaders + uniforms. Open with **Voilà**.

In [ ]:
%%html
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/deecshader.js"></script>
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/deecapp.js"></script>


In [ ]:
%%html
<script id="my-vertex-shader" type="x-shader/x-vertex">
precision highp float;
attribute vec2 in_Position;
attribute vec3 in_Color;
varying vec3 ex_Color;
void main(void){
  gl_Position = vec4(in_Position.x, in_Position.y, 0.0, 1.0);
  ex_Color = in_Color;
}
</script>
<script id="my-fragment-shader" type="x-shader/x-fragment">
precision highp float;
varying vec3 ex_Color;
void main(void){ gl_FragColor = vec4(ex_Color,1.0); }
</script>

<script id="vert-31" type="x-shader/x-vertex">
precision highp float;
attribute vec2 in_Position;
attribute vec3 in_Color;
varying vec3 ex_Color;
void main(void){
  // Ex 3.1: scale by 0.5
  gl_Position = vec4(in_Position.x * 0.5, in_Position.y * 0.5, 0.0, 1.0);
  ex_Color = in_Color;
}
</script>
<script id="vert-32" type="x-shader/x-vertex">
precision highp float;
attribute vec2 in_Position;
attribute vec3 in_Color;
varying vec3 ex_Color;
void main(void){
  gl_Position = vec4(in_Position.x, in_Position.y, 0.0, 1.0);
  // Ex 3.2: red always 1
  ex_Color = vec3(1.0, in_Color.g, in_Color.b);
}
</script>
<script id="frag-33" type="x-shader/x-fragment">
precision highp float;
varying vec3 ex_Color;
void main(void){
  // Ex 3.3: blue threshold
  float blue = ex_Color.b < 0.5 ? 0.0 : 1.0;
  gl_FragColor = vec4(ex_Color.r, ex_Color.g, blue, 1.0);
}
</script>
<script id="vert-34" type="x-shader/x-vertex">
precision highp float;
attribute vec2 in_Position;
attribute vec3 in_Color;
varying vec3 ex_Color;
uniform float timepar;
void main(void){
  float K1 = 0.2;
  float K2 = 500.0;
  float y = in_Position.y + K1 * sin(timepar / K2);
  gl_Position = vec4(in_Position.x, y, 0.0, 1.0);
  ex_Color = in_Color;
}
</script>
<script id="vert-35" type="x-shader/x-vertex">
precision highp float;
attribute vec2 in_Position;
attribute vec3 in_Color;
varying vec3 ex_Color;
uniform float timepar;
void main(void){
  float K2 = 500.0;
  float scale = 0.5 + 0.25 * sin(timepar / K2);
  gl_Position = vec4(in_Position.x * scale, in_Position.y * scale, 0.0, 1.0);
  ex_Color = in_Color;
}
</script>
<script id="vert-36" type="x-shader/x-vertex">
precision highp float;
attribute vec2 in_Position;
attribute vec3 in_Color;
varying vec3 ex_Color;
uniform float timepar;
void main(void){
  float ang = timepar / 1000.0;
  float c = cos(ang);
  float s = sin(ang);
  float x = in_Position.x * c - in_Position.y * s;
  float y = in_Position.x * s + in_Position.y * c;
  gl_Position = vec4(x, y, 0.0, 1.0);
  ex_Color = in_Color;
}
</script>
<script id="frag-37" type="x-shader/x-fragment">
precision highp float;
varying vec3 ex_Color;
uniform float timepar;
void main(void){
  float K2 = 500.0;
  float g = 0.5 + 0.5 * sin(timepar / K2);
  gl_FragColor = vec4(ex_Color.r, g, ex_Color.b, 1.0);
}
</script>


## Ex 3.1 — vertex shader scale 0.5

In [ ]:
%%html
<canvas id="c31" width="400" height="400" style="border:2px solid #000;"></canvas>
<script>

function initCustom(app, vertId, fragId){
  var vertsrc = document.getElementById(vertId).text;
  var fragsrc = document.getElementById(fragId).text;
  app.shaderprog = new DEECshader(app.gl);
  app.shaderprog.srcShaders(vertsrc, fragsrc);
  app.gl.enable(app.gl.DEPTH_TEST);
  app.gl.clearColor(1,1,1,1);
}
function makeBuf(gl, data){
  var b = gl.createBuffer();
  gl.bindBuffer(gl.ARRAY_BUFFER, b);
  gl.bufferData(gl.ARRAY_BUFFER, new Float32Array(data), gl.STATIC_DRAW);
  return b;
}
function draw2(app, vbuf, cbuf, n, mode){
  var gl = app.gl;
  gl.bindBuffer(gl.ARRAY_BUFFER, vbuf);
  var lp = gl.getAttribLocation(app.shaderprog.shaderProgram, 'in_Position');
  gl.vertexAttribPointer(lp, 2, gl.FLOAT, false, 0, 0);
  gl.enableVertexAttribArray(lp);
  gl.bindBuffer(gl.ARRAY_BUFFER, cbuf);
  var lc = gl.getAttribLocation(app.shaderprog.shaderProgram, 'in_Color');
  gl.vertexAttribPointer(lc, 3, gl.FLOAT, false, 0, 0);
  gl.enableVertexAttribArray(lc);
  app.shaderprog.startUsing();
  gl.drawArrays(mode, 0, n);
  app.shaderprog.stopUsing();
}

class App31 extends DEECapp{
  initialize(){
    initCustom(this,'vert-31','my-fragment-shader');
    this.vbuf = makeBuf(this.gl, [0.0,0.7, -0.6,-0.5, 0.6,-0.5]);
    this.cbuf = makeBuf(this.gl, [1,0,0, 0,1,0, 0,0,1]);
  }
  render(){
    this.gl.clear(this.gl.COLOR_BUFFER_BIT);
    draw2(this,this.vbuf,this.cbuf,3,this.gl.TRIANGLES);
  }
}
new App31('c31').run();
</script>


## Ex 3.2 — red component always 1

In [ ]:
%%html
<canvas id="c32" width="400" height="400" style="border:2px solid #000;"></canvas>
<script>

function initCustom(app, vertId, fragId){
  var vertsrc = document.getElementById(vertId).text;
  var fragsrc = document.getElementById(fragId).text;
  app.shaderprog = new DEECshader(app.gl);
  app.shaderprog.srcShaders(vertsrc, fragsrc);
  app.gl.enable(app.gl.DEPTH_TEST);
  app.gl.clearColor(1,1,1,1);
}
function makeBuf(gl, data){
  var b = gl.createBuffer();
  gl.bindBuffer(gl.ARRAY_BUFFER, b);
  gl.bufferData(gl.ARRAY_BUFFER, new Float32Array(data), gl.STATIC_DRAW);
  return b;
}
function draw2(app, vbuf, cbuf, n, mode){
  var gl = app.gl;
  gl.bindBuffer(gl.ARRAY_BUFFER, vbuf);
  var lp = gl.getAttribLocation(app.shaderprog.shaderProgram, 'in_Position');
  gl.vertexAttribPointer(lp, 2, gl.FLOAT, false, 0, 0);
  gl.enableVertexAttribArray(lp);
  gl.bindBuffer(gl.ARRAY_BUFFER, cbuf);
  var lc = gl.getAttribLocation(app.shaderprog.shaderProgram, 'in_Color');
  gl.vertexAttribPointer(lc, 3, gl.FLOAT, false, 0, 0);
  gl.enableVertexAttribArray(lc);
  app.shaderprog.startUsing();
  gl.drawArrays(mode, 0, n);
  app.shaderprog.stopUsing();
}

class App32 extends DEECapp{
  initialize(){
    initCustom(this,'vert-32','my-fragment-shader');
    this.vbuf = makeBuf(this.gl, [0.0,0.7, -0.6,-0.5, 0.6,-0.5]);
    this.cbuf = makeBuf(this.gl, [1,0,0, 0,1,0, 0,0,1]);
  }
  render(){
    this.gl.clear(this.gl.COLOR_BUFFER_BIT);
    draw2(this,this.vbuf,this.cbuf,3,this.gl.TRIANGLES);
  }
}
new App32('c32').run();
</script>


## Ex 3.3 — fragment blue threshold

In [ ]:
%%html
<canvas id="c33" width="400" height="400" style="border:2px solid #000;"></canvas>
<script>

function initCustom(app, vertId, fragId){
  var vertsrc = document.getElementById(vertId).text;
  var fragsrc = document.getElementById(fragId).text;
  app.shaderprog = new DEECshader(app.gl);
  app.shaderprog.srcShaders(vertsrc, fragsrc);
  app.gl.enable(app.gl.DEPTH_TEST);
  app.gl.clearColor(1,1,1,1);
}
function makeBuf(gl, data){
  var b = gl.createBuffer();
  gl.bindBuffer(gl.ARRAY_BUFFER, b);
  gl.bufferData(gl.ARRAY_BUFFER, new Float32Array(data), gl.STATIC_DRAW);
  return b;
}
function draw2(app, vbuf, cbuf, n, mode){
  var gl = app.gl;
  gl.bindBuffer(gl.ARRAY_BUFFER, vbuf);
  var lp = gl.getAttribLocation(app.shaderprog.shaderProgram, 'in_Position');
  gl.vertexAttribPointer(lp, 2, gl.FLOAT, false, 0, 0);
  gl.enableVertexAttribArray(lp);
  gl.bindBuffer(gl.ARRAY_BUFFER, cbuf);
  var lc = gl.getAttribLocation(app.shaderprog.shaderProgram, 'in_Color');
  gl.vertexAttribPointer(lc, 3, gl.FLOAT, false, 0, 0);
  gl.enableVertexAttribArray(lc);
  app.shaderprog.startUsing();
  gl.drawArrays(mode, 0, n);
  app.shaderprog.stopUsing();
}

class App33 extends DEECapp{
  initialize(){
    initCustom(this,'my-vertex-shader','frag-33');
    this.vbuf = makeBuf(this.gl, [0.0,0.7, -0.6,-0.5, 0.6,-0.5]);
    this.cbuf = makeBuf(this.gl, [1,0,0, 0,1,0, 0,0,1]);
  }
  render(){
    this.gl.clear(this.gl.COLOR_BUFFER_BIT);
    draw2(this,this.vbuf,this.cbuf,3,this.gl.TRIANGLES);
  }
}
new App33('c33').run();
</script>


## Ex 3.4 — y += K1*sin(time/K2) via uniform

In [ ]:
%%html
<canvas id="c34" width="400" height="400" style="border:2px solid #000;"></canvas>
<script>

function initCustom(app, vertId, fragId){
  var vertsrc = document.getElementById(vertId).text;
  var fragsrc = document.getElementById(fragId).text;
  app.shaderprog = new DEECshader(app.gl);
  app.shaderprog.srcShaders(vertsrc, fragsrc);
  app.gl.enable(app.gl.DEPTH_TEST);
  app.gl.clearColor(1,1,1,1);
}
function makeBuf(gl, data){
  var b = gl.createBuffer();
  gl.bindBuffer(gl.ARRAY_BUFFER, b);
  gl.bufferData(gl.ARRAY_BUFFER, new Float32Array(data), gl.STATIC_DRAW);
  return b;
}
function draw2(app, vbuf, cbuf, n, mode){
  var gl = app.gl;
  gl.bindBuffer(gl.ARRAY_BUFFER, vbuf);
  var lp = gl.getAttribLocation(app.shaderprog.shaderProgram, 'in_Position');
  gl.vertexAttribPointer(lp, 2, gl.FLOAT, false, 0, 0);
  gl.enableVertexAttribArray(lp);
  gl.bindBuffer(gl.ARRAY_BUFFER, cbuf);
  var lc = gl.getAttribLocation(app.shaderprog.shaderProgram, 'in_Color');
  gl.vertexAttribPointer(lc, 3, gl.FLOAT, false, 0, 0);
  gl.enableVertexAttribArray(lc);
  app.shaderprog.startUsing();
  gl.drawArrays(mode, 0, n);
  app.shaderprog.stopUsing();
}

class App34 extends DEECapp{
  initialize(){
    initCustom(this,'vert-34','my-fragment-shader');
    this.vbuf = makeBuf(this.gl, [0.0,0.7, -0.6,-0.5, 0.6,-0.5]);
    this.cbuf = makeBuf(this.gl, [1,0,0, 0,1,0, 0,0,1]);
    this.t0 = Date.now();
    this.timeLoc = null;
  }
  render(){
    var timevariable = (Date.now()-this.t0)*1.0;
    this.shaderprog.startUsing();
    if(this.timeLoc===null)
      this.timeLoc = this.gl.getUniformLocation(this.shaderprog.shaderProgram,'timepar');
    this.gl.uniform1f(this.timeLoc, timevariable);
    this.shaderprog.stopUsing();
    this.gl.clear(this.gl.COLOR_BUFFER_BIT);
    draw2(this,this.vbuf,this.cbuf,3,this.gl.TRIANGLES);
  }
}
new App34('c34').run();
</script>


## Ex 3.5 — pulsing scale with time

In [ ]:
%%html
<canvas id="c35" width="400" height="400" style="border:2px solid #000;"></canvas>
<script>

function initCustom(app, vertId, fragId){
  var vertsrc = document.getElementById(vertId).text;
  var fragsrc = document.getElementById(fragId).text;
  app.shaderprog = new DEECshader(app.gl);
  app.shaderprog.srcShaders(vertsrc, fragsrc);
  app.gl.enable(app.gl.DEPTH_TEST);
  app.gl.clearColor(1,1,1,1);
}
function makeBuf(gl, data){
  var b = gl.createBuffer();
  gl.bindBuffer(gl.ARRAY_BUFFER, b);
  gl.bufferData(gl.ARRAY_BUFFER, new Float32Array(data), gl.STATIC_DRAW);
  return b;
}
function draw2(app, vbuf, cbuf, n, mode){
  var gl = app.gl;
  gl.bindBuffer(gl.ARRAY_BUFFER, vbuf);
  var lp = gl.getAttribLocation(app.shaderprog.shaderProgram, 'in_Position');
  gl.vertexAttribPointer(lp, 2, gl.FLOAT, false, 0, 0);
  gl.enableVertexAttribArray(lp);
  gl.bindBuffer(gl.ARRAY_BUFFER, cbuf);
  var lc = gl.getAttribLocation(app.shaderprog.shaderProgram, 'in_Color');
  gl.vertexAttribPointer(lc, 3, gl.FLOAT, false, 0, 0);
  gl.enableVertexAttribArray(lc);
  app.shaderprog.startUsing();
  gl.drawArrays(mode, 0, n);
  app.shaderprog.stopUsing();
}

class App35 extends DEECapp{
  initialize(){
    initCustom(this,'vert-35','my-fragment-shader');
    this.vbuf = makeBuf(this.gl, [0.0,0.7, -0.6,-0.5, 0.6,-0.5]);
    this.cbuf = makeBuf(this.gl, [1,0,0, 0,1,0, 0,0,1]);
    this.t0 = Date.now();
    this.timeLoc = null;
  }
  render(){
    var timevariable = (Date.now()-this.t0)*1.0;
    this.shaderprog.startUsing();
    if(this.timeLoc===null)
      this.timeLoc = this.gl.getUniformLocation(this.shaderprog.shaderProgram,'timepar');
    this.gl.uniform1f(this.timeLoc, timevariable);
    this.shaderprog.stopUsing();
    this.gl.clear(this.gl.COLOR_BUFFER_BIT);
    draw2(this,this.vbuf,this.cbuf,3,this.gl.TRIANGLES);
  }
}
new App35('c35').run();
</script>


## Ex 3.6 — rotate in vertex shader with time

In [ ]:
%%html
<canvas id="c36" width="400" height="400" style="border:2px solid #000;"></canvas>
<script>

function initCustom(app, vertId, fragId){
  var vertsrc = document.getElementById(vertId).text;
  var fragsrc = document.getElementById(fragId).text;
  app.shaderprog = new DEECshader(app.gl);
  app.shaderprog.srcShaders(vertsrc, fragsrc);
  app.gl.enable(app.gl.DEPTH_TEST);
  app.gl.clearColor(1,1,1,1);
}
function makeBuf(gl, data){
  var b = gl.createBuffer();
  gl.bindBuffer(gl.ARRAY_BUFFER, b);
  gl.bufferData(gl.ARRAY_BUFFER, new Float32Array(data), gl.STATIC_DRAW);
  return b;
}
function draw2(app, vbuf, cbuf, n, mode){
  var gl = app.gl;
  gl.bindBuffer(gl.ARRAY_BUFFER, vbuf);
  var lp = gl.getAttribLocation(app.shaderprog.shaderProgram, 'in_Position');
  gl.vertexAttribPointer(lp, 2, gl.FLOAT, false, 0, 0);
  gl.enableVertexAttribArray(lp);
  gl.bindBuffer(gl.ARRAY_BUFFER, cbuf);
  var lc = gl.getAttribLocation(app.shaderprog.shaderProgram, 'in_Color');
  gl.vertexAttribPointer(lc, 3, gl.FLOAT, false, 0, 0);
  gl.enableVertexAttribArray(lc);
  app.shaderprog.startUsing();
  gl.drawArrays(mode, 0, n);
  app.shaderprog.stopUsing();
}

class App36 extends DEECapp{
  initialize(){
    initCustom(this,'vert-36','my-fragment-shader');
    this.vbuf = makeBuf(this.gl, [0.0,0.7, -0.6,-0.5, 0.6,-0.5]);
    this.cbuf = makeBuf(this.gl, [1,0,0, 0,1,0, 0,0,1]);
    this.t0 = Date.now();
    this.timeLoc = null;
  }
  render(){
    var timevariable = (Date.now()-this.t0)*1.0;
    this.shaderprog.startUsing();
    if(this.timeLoc===null)
      this.timeLoc = this.gl.getUniformLocation(this.shaderprog.shaderProgram,'timepar');
    this.gl.uniform1f(this.timeLoc, timevariable);
    this.shaderprog.stopUsing();
    this.gl.clear(this.gl.COLOR_BUFFER_BIT);
    draw2(this,this.vbuf,this.cbuf,3,this.gl.TRIANGLES);
  }
}
new App36('c36').run();
</script>


## Ex 3.7 — green varies with time in fragment shader

In [ ]:
%%html
<canvas id="c37" width="400" height="400" style="border:2px solid #000;"></canvas>
<script>

function initCustom(app, vertId, fragId){
  var vertsrc = document.getElementById(vertId).text;
  var fragsrc = document.getElementById(fragId).text;
  app.shaderprog = new DEECshader(app.gl);
  app.shaderprog.srcShaders(vertsrc, fragsrc);
  app.gl.enable(app.gl.DEPTH_TEST);
  app.gl.clearColor(1,1,1,1);
}
function makeBuf(gl, data){
  var b = gl.createBuffer();
  gl.bindBuffer(gl.ARRAY_BUFFER, b);
  gl.bufferData(gl.ARRAY_BUFFER, new Float32Array(data), gl.STATIC_DRAW);
  return b;
}
function draw2(app, vbuf, cbuf, n, mode){
  var gl = app.gl;
  gl.bindBuffer(gl.ARRAY_BUFFER, vbuf);
  var lp = gl.getAttribLocation(app.shaderprog.shaderProgram, 'in_Position');
  gl.vertexAttribPointer(lp, 2, gl.FLOAT, false, 0, 0);
  gl.enableVertexAttribArray(lp);
  gl.bindBuffer(gl.ARRAY_BUFFER, cbuf);
  var lc = gl.getAttribLocation(app.shaderprog.shaderProgram, 'in_Color');
  gl.vertexAttribPointer(lc, 3, gl.FLOAT, false, 0, 0);
  gl.enableVertexAttribArray(lc);
  app.shaderprog.startUsing();
  gl.drawArrays(mode, 0, n);
  app.shaderprog.stopUsing();
}

class App37 extends DEECapp{
  initialize(){
    initCustom(this,'my-vertex-shader','frag-37');
    this.vbuf = makeBuf(this.gl, [0.0,0.7, -0.6,-0.5, 0.6,-0.5]);
    this.cbuf = makeBuf(this.gl, [1,0,0, 0,1,0, 0,0,1]);
    this.t0 = Date.now();
    this.timeLoc = null;
  }
  render(){
    var timevariable = (Date.now()-this.t0)*1.0;
    this.shaderprog.startUsing();
    if(this.timeLoc===null)
      this.timeLoc = this.gl.getUniformLocation(this.shaderprog.shaderProgram,'timepar');
    this.gl.uniform1f(this.timeLoc, timevariable);
    this.shaderprog.stopUsing();
    this.gl.clear(this.gl.COLOR_BUFFER_BIT);
    draw2(this,this.vbuf,this.cbuf,3,this.gl.TRIANGLES);
  }
}
new App37('c37').run();
</script>
